# Tool Use with Claude Agents in AG2

This notebook demonstrates how to give Claude-powered AG2 agents the ability to call Python functions using AG2's decorator-based tool registration.

## What you'll learn
- How to define tools as Python functions
- How to register tools with AG2's decorator pattern
- How Claude agents discover and call tools automatically

In [1]:
%pip install "ag2[anthropic]>=0.11.4,<1.0" -q

/Users/faridunm/Documents/WORK/AG2/Opensource/claude-cookbooks/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from typing import Annotated

from autogen import AssistantAgent, LLMConfig, UserProxyAgent

llm_config = LLMConfig(
    {
        "model": "claude-sonnet-4-6",
        "api_key": os.environ.get("ANTHROPIC_API_KEY"),
        "api_type": "anthropic",
    }
)

assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful assistant with access to tools. "
        "Use the available tools to answer questions accurately. "
        "Reply TERMINATE when done."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

## Defining and Registering Tools

AG2 uses a decorator pattern for tool registration:
- `@user_proxy.register_for_execution()` — tells the UserProxy to execute this function when called
- `@assistant.register_for_llm(description="...")` — tells the Assistant this tool is available

Claude will automatically discover registered tools and call them when appropriate.

In [3]:
@user_proxy.register_for_execution()
@assistant.register_for_llm(description="Get the current weather for a given city")
def get_weather(
    city: Annotated[str, "The city name to get weather for"],
) -> str:
    """Simulated weather lookup."""
    weather_data = {
        "San Francisco": "62\u00b0F, Foggy",
        "New York": "75\u00b0F, Sunny",
        "London": "58\u00b0F, Rainy",
        "Tokyo": "80\u00b0F, Humid",
    }
    return weather_data.get(city, f"Weather data not available for {city}")


@user_proxy.register_for_execution()
@assistant.register_for_llm(description="Convert temperature between Fahrenheit and Celsius")
def convert_temperature(
    value: Annotated[float, "The temperature value to convert"],
    from_unit: Annotated[str, "Source unit: 'F' or 'C'"],
) -> str:
    """Convert temperature between Fahrenheit and Celsius."""
    if from_unit.upper() == "F":
        result = (value - 32) * 5 / 9
        return f"{value}\u00b0F = {result:.1f}\u00b0C"
    elif from_unit.upper() == "C":
        result = value * 9 / 5 + 32
        return f"{value}\u00b0C = {result:.1f}\u00b0F"
    return f"Unknown unit: {from_unit}"

In [4]:
chat = user_proxy.run(
    assistant,
    message="What's the weather in Tokyo? Also, convert that temperature to Celsius.",
)
chat.process()

User (to Assistant):



What's the weather in Tokyo? Also, convert that temperature to Celsius.



--------------------------------------------------------------------------------


Assistant (to User):



Sure! Let me start by fetching the current weather in Tokyo first.


...


--------------------------------------------------------------------------------


Assistant (to User):



Here's the current weather in **Tokyo**:

- 🌡️ **Temperature:** 80°F (**26.7°C**)
- 💧 **Condition:** Humid

It's a warm and humid day in Tokyo! TERMINATE



--------------------------------------------------------------------------------



>>>>>>>> TERMINATING RUN (ad0f76c5-1c6a-4055-b411-a94385ae86f9): Termination message condition on agent 'User' met


## How It Works

1. The user sends a message to the assistant
2. Claude analyzes the message and decides which tools to call
3. The UserProxy executes the tool and returns the result
4. Claude incorporates the tool result into its response
5. The cycle continues until the task is complete

This pattern maps naturally to Claude's native tool use capabilities, with AG2 handling the orchestration layer.

## Next Steps

- [GroupChat_Orchestration.ipynb](./GroupChat_Orchestration.ipynb) — Orchestrate multiple specialized agents
- [AG2 Tool Use Documentation](https://docs.ag2.ai/docs/tutorial/tool-use) — Advanced tool patterns